### Tools
- Models can request to call tools that perform tasks such as fectching data from a database , searching the web or running code.Tools are a pairing of :
1. A schema , including the name of the tool , a descrption and/or argument definitions(often a JSON schema)
2. A function or coroutine to execute

In [ ]:
import os

from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke(" why do parrots talk?"
)
response

In [ ]:
from langchain.tools import tool

@tool # converts a Python function into a LangChain Tool that an LLM agent can discover and invoke.
def get_weather(location:str)->str:
    """Get the weather at a location""" #
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("what's the weather like in Boston?")

for tool_call in response.tool_calls:
    # View the tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

print(response)


### Tool Execution Loops

In [ ]:
# Step 1 : Model generates tool calls
messages = [{"role": "user" , "content": " what's the weather in Boston?"}];
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)



# Step 2: Execute tools and collect results
for tool_call in ai_message.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

print(messages)

# step 3 : Pass results back to model for final response
final_response = model_with_tools.invoke(messages)

print (final_response.text)